<a href="https://colab.research.google.com/github/MithunSrinivas28/wafer-defect-ai/blob/main/Wafer_detect.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### **Mount Google Drive**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


### Load Dataset Using TensorFlow

In [ ]:
TRAIN_PATH = "/content/drive/Shared-With-Me/Datasets/train"
TEST_PATH  = "/content/drive/Shared-With-Me/Datasets/test"


In [ ]:
import tensorflow as tf

TRAIN_PATH = "/content/drive/MyDrive/Datasets/train"
TEST_PATH  = "/content/drive/MyDrive/Datasets/test"

IMG_SIZE = (224, 224)
BATCH_SIZE = 32

train_data = tf.keras.preprocessing.image_dataset_from_directory(
    TRAIN_PATH,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=True
)

test_data = tf.keras.preprocessing.image_dataset_from_directory(
    TEST_PATH,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=False
)


Found 585 files belonging to 8 classes.
Found 43 files belonging to 8 classes.


In [ ]:
import tensorflow as tf

# Reload only to get class names
temp_ds = tf.keras.preprocessing.image_dataset_from_directory(
    "/content/drive/MyDrive/Datasets/train",
    image_size=(224,224),
    batch_size=32
)

class_names = temp_ds.class_names
NUM_CLASSES = len(class_names)

print("Classes:", class_names)
print("Num classes:", NUM_CLASSES)


Found 585 files belonging to 8 classes.
Classes: ['bridge', 'clean', 'cmp', 'crack', 'ler', 'open', 'others', 'vias']
Num classes: 8


In [ ]:
import os

for c in os.listdir(TRAIN_PATH):
    print(c, len(os.listdir(TRAIN_PATH + "/" + c)))


bridge 87
clean 57
cmp 72
crack 112
ler 41
open 51
vias 101
others 64


### Normalize + Add Data Augmentation

In [ ]:
from tensorflow.keras import layers

# Normalize (0–255 → 0–1)
normalization = layers.Rescaling(1./255)

# Data Augmentation
#data_augmentation = tf.keras.Sequential([
 #   layers.RandomFlip("horizontal"),
   # layers.RandomRotation(0.2),
   # layers.RandomZoom(0.2),
   # layers.RandomContrast(0.2),
#])
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
])

# Apply to datasets
train_data = train_data.map(lambda x, y: (normalization(data_augmentation(x)), y))
test_data  = test_data.map(lambda x, y: (normalization(x), y))


### Build the MobileNet Model (Your AI Brain)

In [ ]:
from tensorflow.keras import layers, models
import tensorflow as tf

base_model = tf.keras.applications.MobileNetV3Small(
    input_shape=(224,224,3),
    include_top=False,
    #weights="imagenet"
    weights=None
  )

base_model.trainable = True

model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    #layers.Dense(128, activation="relu"),
    #layers.Dropout(0.3),
    layers.Dense(NUM_CLASSES, activation="softmax")
])

model.summary()


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ MobileNetV3Small (Functional)   │ (None, 7, 7, 576)      │       939,120 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 576)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 8)              │         4,616 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 943,736 (3.60 MB)

 Trainable params: 931,624 (3.55 MB)

 Non-trainable params: 12,112 (47.31 KB)

##** Compile the Model**

In [ ]:
model.compile(
    #optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    #loss="sparse_categorical_crossentropy",
   optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

In [ ]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

y = []
for _, labels in train_data:
    y.extend(labels.numpy())

class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y),
    y=y
)

class_weights = dict(enumerate(class_weights))
print(class_weights)


{0: np.float64(0.8405172413793104), 1: np.float64(1.2828947368421053), 2: np.float64(1.015625), 3: np.float64(0.6529017857142857), 4: np.float64(1.7835365853658536), 5: np.float64(1.4338235294117647), 6: np.float64(1.142578125), 7: np.float64(0.724009900990099)}


### Train the Model

In [ ]:
EPOCHS = 15   # good for small dataset

history = model.fit(
    train_data,
    validation_data=test_data,
    epochs=EPOCHS,
    class_weight=class_weights
)


Epoch 1/15
19/19 ━━━━━━━━━━━━━━━━━━━━ 93s 3s/step - accuracy: 0.3178 - loss: 1.8026 - val_accuracy: 0.1628 - val_loss: 2.0804
Epoch 2/15
19/19 ━━━━━━━━━━━━━━━━━━━━ 3s 160ms/step - accuracy: 0.5945 - loss: 1.0974 - val_accuracy: 0.1628 - val_loss: 2.0798
Epoch 3/15
19/19 ━━━━━━━━━━━━━━━━━━━━ 3s 172ms/step - accuracy: 0.6672 - loss: 0.9307 - val_accuracy: 0.1628 - val_loss: 2.0792
Epoch 4/15
19/19 ━━━━━━━━━━━━━━━━━━━━ 4s 197ms/step - accuracy: 0.7176 - loss: 0.7220 - val_accuracy: 0.1628 - val_loss: 2.0783
Epoch 5/15
19/19 ━━━━━━━━━━━━━━━━━━━━ 5s 168ms/step - accuracy: 0.6944 - loss: 0.8413 - val_accuracy: 0.1628 - val_loss: 2.0794
Epoch 6/15
19/19 ━━━━━━━━━━━━━━━━━━━━ 5s 157ms/step - accuracy: 0.8041 - loss: 0.6152 - val_accuracy: 0.1628 - val_loss: 2.0801
Epoch 7/15
19/19 ━━━━━━━━━━━━━━━━━━━━ 6s 226ms/step - accuracy: 0.8288 - loss: 0.4883 - val_accuracy: 0.1628 - val_loss: 2.0794
Epoch 8/15
19/19 ━━━━━━━━━━━━━━━━━━━━ 4s 171ms/step - accuracy: 0.8224 - loss: 0.5128 - val_accuracy: 0.16

In [ ]:
model.evaluate(train_data)
model.evaluate(test_data)
#model is overfitting

19/19 ━━━━━━━━━━━━━━━━━━━━ 7s 379ms/step - accuracy: 0.0990 - loss: 2.0735
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 84ms/step - accuracy: 0.1814 - loss: 2.0770


[2.0870909690856934, 0.1627907007932663]